# Concurrency with Python

- Concurrency: Is doing multiple computation tasks at a time, while 
    - Code that can be executed out of order.
    - Juggling multiple tasks by interleaving execution. Tasks take turns.
    - Accessing the same resources, e.g. CPU cores or disk, "simultaneously"
    - Python package: `threading` , `asyncio`
- Parallelism:
    - Capability to execute code simultaneously.
    - Executing multiple tasks at the exact same physical instant.
    - Multiple tasks accessing separate resources
    - Python package: `multiprocessing`


- Global Interpreter Lock (GIL)
    - GIL locks the execution of python bytecode itself. Only one thread runs bytecode at a time.
    - The GIL prevents two threads from executing bytecode simultaneously, but it does not prevent a thread from being interrupted mid-operation between bytecode instructions.
    - GIL can interleave in the middle of a multi-step operation and hence may result in race condition bugs. These are bugs that causes two threads reading/modifying shared state with unsynchronized interleaving to cause unintended modifications to the shared variable.

- 

- Managing Race condition in Python while Multithreading using - `threading.Lock()`
    - This is a mutual-exclusion lock or a gatekeeper object.
    - Only one thread can "hold" it at a time. Any thread trying to acquire a lock that's already held simply blocks (waits) until the holder releases it.

        ```python
        lock = threading.Lock()

        lock.acquire()
        # critical section — only one thread can be here at a time
        lock.release()
        ```
    - Anything between `acquire()` and `release()` is called a critical section — code that touches shared state and must not be interrupted by another thread mid-way through.
    - The problem with `acquire`/`release` directly: if an exception happens inside the critical section, `release()` never gets called, and the lock stays held forever (every other thread blocks forever too — a deadlock). So in practice, you almost always use the lock as a context manager instead:
    ```python
    lock = threading.Lock()

    with lock:
        # critical section
        counter += 1
    # lock is automatically released here, even if an exception occurred
    ```
    - This is the same with pattern you'd use for file handling — guarantees cleanup regardless of what happens inside.
    - Even if the GIL switches to the other thread mid-instruction, that other thread hits `with lock:` and blocks immediately, because the first thread still holds it. No interleaving can happen inside the critical section anymore.

- One trade-off worth naming in an interview: locks reintroduce the very serialization you were trying to avoid by using threads — this critical section is now effectively single-threaded again. That's fine for a short increment, but if your critical section is large, you lose most of the concurrency benefit. This is also why people reach for multiprocessing for CPU-bound work in the first place, rather than "just add locks everywhere."

- When to use multi-processing and when to use multi-threading?
    - When the multithreaded operation is bound by multiple locks that essentially convert the concurrent operation into a serialized execution pattern, it is recommended to switch to `multiprocessing`.


In [1]:
import threading

counter = 0

def increment():
    global counter
    for _ in range(100000):
        counter += 1

threads = [threading.Thread(target=increment) for _ in range(2)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(counter)

200000


`.join()` guarantees the main thread waits for both threads to finish before printing. But it says nothing about what happens while they're running concurrently, or whether their operations on counter interfere with each other.